# Notebook 04 – Prompt Engineering Experiments (Part C)

Run the same 5 queries through three prompt variants and compare outputs.
Record observations manually in `experiment_logs/partC_prompting.md`.

In [1]:
%pip install -q pandas pypdf numpy sentence-transformers openai tiktoken python-dotenv scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.join('..', 'backend'))
from pathlib import Path
from raghana.pipeline import run as rag_run

QUERIES = [
    'How many votes did NPP receive in Ashanti Region in 2020?',
    "What is Ghana's debt-to-GDP target for 2025?",
    'Who won the election in Greater Accra in 2016?',
    'What are the revenue targets in the 2025 budget?',
    'How much money did the NDC spend on free SHS?',  # misleading — should trigger refusal
]

## 4.1 Variant v1 — Naïve baseline

In [3]:
print('=== PROMPT v1 (naïve baseline) ===')
v1_results = {}
for q in QUERIES:
    r = rag_run(q, prompt_version='v1')
    v1_results[q] = r.answer
    print(f'\nQ: {q}')
    print(f'A: {r.answer}')
    print('-' * 60)

2026-04-24 16:23:28,996 [INFO] raghana: Use pytorch device_name: cpu


2026-04-24 16:23:28,997 [INFO] raghana: Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


=== PROMPT v1 (naïve baseline) ===


2026-04-24 16:23:35,642 [INFO] raghana: [retrieve] 6675ms | 5 chunks | top_score=0.032 | mode=hybrid


2026-04-24 16:23:35,644 [INFO] raghana: [multistep] 0ms | Total NPP votes across all retrieved regions/years (1992, 1996, 2000, 2008, 2020


...

2026-04-24 16:23:45,836 [INFO] raghana: [prompt_build] 5ms | 7643 chars



Q: What are the revenue targets in the 2025 budget?
A: The revenue targets in the 2025 budget are as follows:

- Total Revenue and Grants are projected to moderate from 15.9 percent of GDP in 2024 to 16.1 percent in 2025, and further to 16.9 percent by 2028.

This improvement in revenue is expected to be driven by a mix of tax policy measures, strengthened revenue administration, and an improvement in collection efficiency and tax compliance.
------------------------------------------------------------


2026-04-24 16:23:47,751 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-04-24 16:23:47,755 [INFO] raghana: [generate] 1917ms | The provided text does not specify the amount spent by the National Democratic C



Q: How much money did the NDC spend on free SHS?
A: The provided text does not specify the amount spent by the National Democratic Congress (NDC) on the Free Senior High School (SHS) program. To find that information, you may need to refer to specific budget documents or reports from the NDC or the Ministry of Finance that detail expenditures on the Free SHS initiative.
------------------------------------------------------------


## 4.2 Variant v2 — Grounded + citation (default)

In [4]:
print('=== PROMPT v2 (grounded + citation) ===')
v2_results = {}
for q in QUERIES:
    r = rag_run(q, prompt_version='v2')
    v2_results[q] = r.answer
    print(f'\nQ: {q}')
    print(f'A: {r.answer}')
    print('-' * 60)

2026-04-24 16:23:47,797 [INFO] raghana: [retrieve] 32ms | 5 chunks | top_score=0.032 | mode=hybrid


2026-04-24 16:23:47,799 [INFO] raghana: [multistep] 1ms | Total NPP votes across all retrieved regions/years (1992, 1996, 2000, 2008, 2020


2026-04-24 16:23:47,802 [INFO] raghana: [prompt_build] 1ms | 2795 chars


=== PROMPT v2 (grounded + citation) ===


2026-04-24 16:23:49,325 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

2026-04-24 16:23:53,725 [INFO] raghana: [prompt_build] 4ms | 8170 chars



Q: What are the revenue targets in the 2025 budget?
A: I do not have that information in the provided documents.
------------------------------------------------------------


2026-04-24 16:23:54,711 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-04-24 16:23:54,713 [INFO] raghana: [generate] 987ms | I do not have that information in the provided documents.



Q: How much money did the NDC spend on free SHS?
A: I do not have that information in the provided documents.
------------------------------------------------------------


## 4.3 Variant v3 — Grounded + citation + refusal + few-shot

In [5]:
print('=== PROMPT v3 (grounded + citation + refusal + few-shot) ===')
v3_results = {}
for q in QUERIES:
    r = rag_run(q, prompt_version='v3')
    v3_results[q] = r.answer
    print(f'\nQ: {q}')
    print(f'A: {r.answer}')
    print('-' * 60)

2026-04-24 16:23:54,767 [INFO] raghana: [retrieve] 42ms | 5 chunks | top_score=0.032 | mode=hybrid


2026-04-24 16:23:54,769 [INFO] raghana: [multistep] 1ms | Total NPP votes across all retrieved regions/years (1992, 1996, 2000, 2008, 2020


2026-04-24 16:23:54,773 [INFO] raghana: [prompt_build] 1ms | 2954 chars


=== PROMPT v3 (grounded + citation + refusal + few-shot) ===


2026-04-24 16:23:55,918 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

2026-04-24 16:24:00,778 [INFO] raghana: [prompt_build] 4ms | 8329 chars



Q: What are the revenue targets in the 2025 budget?
A: Total Revenue and Grants are projected to moderate from 15.9 percent of GDP in 2024 to 16.1 percent in 2025, and further to 16.9 percent by 2028 [C3].
------------------------------------------------------------


2026-04-24 16:24:01,481 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-04-24 16:24:01,483 [INFO] raghana: [generate] 704ms | I do not have that information in the provided documents.



Q: How much money did the NDC spend on free SHS?
A: I do not have that information in the provided documents.
------------------------------------------------------------


## 4.4 Comparison table

After running all cells, complete this table manually:

| Query | v1 cites? | v2 cites? | v3 cites? | v1 hallucinates? | v2 hallucinates? | v3 hallucinates? |
|---|---|---|---|---|---|---|
| NPP Ashanti 2020 | | | | | | |
| Debt-to-GDP 2025 | | | | | | |
| Accra 2016 winner | | | | | | |
| Revenue targets | | | | | | |
| NDC free SHS | | | | | | |

**Manual log:** `experiment_logs/partC_prompting.md`

In [6]:
# Show the final prompt for one query to verify context packing
r = rag_run(QUERIES[0], prompt_version='v2')
print(r.final_prompt)

2026-04-24 16:24:01,534 [INFO] raghana: [retrieve] 41ms | 5 chunks | top_score=0.032 | mode=hybrid


2026-04-24 16:24:01,536 [INFO] raghana: [multistep] 1ms | Total NPP votes across all retrieved regions/years (1992, 1996, 2000, 2008, 2020


2026-04-24 16:24:01,539 [INFO] raghana: [prompt_build] 1ms | 2795 chars


2026-04-24 16:24:02,513 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-04-24 16:24:02,515 [INFO] raghana: [generate] 973ms | NPP received 1,795,824 votes in Ashanti Region in 2020 [C1].


SYSTEM:
You are RAGhana, a factual assistant for Ghana public-sector data (election results and national budget).

Rules:
1. Answer ONLY from the CONTEXT section provided below.
2. If the answer is not present in the context, reply exactly:
   "I do not have that information in the provided documents."
3. Cite supporting chunk IDs inline, e.g. [C1] or [C2].
4. Do not speculate or infer facts that are not explicitly stated.
5. For numeric answers, quote the exact figure from the context.
6. Be concise and factual.

USER:
COMPUTED_RESULT (verified by direct arithmetic on the retrieved data — you MUST quote this figure verbatim in your answer):
Total NPP votes across all retrieved regions/years (1992, 1996, 2000, 2008, 2020): 6,266,614

CONTEXT:
[C1] (csv:2020/Ashanti Region)
In the 2020 presidential election in Ashanti Region:
  Nana Akufo Addo (NPP) received 1,795,824 votes (72.79%)
  John Dramani Mahama (NDC) received 653,149 votes (26.47%)
  Ivor Kobina Greenstreet (CPP) received 1,35